In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib


In [ ]:
df = pd.read_csv("Crop_recommendation.csv")
df = df[['N','P','K','temperature','humidity','ph','rainfall','label']]
df.head()


In [ ]:
X = df.drop('label', axis=1)
y = df['label']


In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    random_state=42
)

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(
    cm,
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.xlabel("Predicted Crop")
plt.ylabel("Actual Crop")
plt.title("Crop Recommendation Confusion Matrix")
plt.show()


In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importance.plot(kind='bar', figsize=(10,4), title="Feature Importance")
plt.show()


In [ ]:
pipeline_bundle = {
    'model': model,
    'scaler': scaler,
    'label_encoder': label_encoder
}

joblib.dump(pipeline_bundle, 'crop_model.joblib', compress=3)
